# Overview of the Encoder Architecture

![encoder_overview](../resources/v0_6/encoder_overview.png)

The encoder architecture is transformer-based with linear attention, SwiGLU and RMSNorm. This notebook will walk through each layer so that the reader can build an intuition for what each layer is doing to the data. To this extent, you'll see that we set the layer initializations and numbers to ones where you can hand calculate if you need to follow a layer better. 

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start with a simple data prep. Here we'll use a small batch of 2 samples, each with 8 gene expression counts. 

In [2]:
batch = 2 # Batch
num_genes = 8 # context, aka num of genes


SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
x_input = torch.from_numpy(np.round(np.random.uniform(1, 5, size=(batch, num_genes)), 0)).float() # [batch, num_genes]
total_counts = torch.from_numpy(np.random.randint(1, 10, size=(batch))).float()# [batch]

x_input.shape, x_input, total_counts.shape, total_counts

(torch.Size([2, 8]),
 tensor([[2., 2., 2., 3., 2., 3., 2., 5.],
         [4., 1., 3., 4., 2., 5., 3., 4.]]),
 torch.Size([2]),
 tensor([2., 4.]))

## Data Masking 
(only done for the Context/Student) We create a mask to make the target prediction task harder. We only evaluate the masked positions when we determine loss. We'll first generate a random distribution and then everything below our target masking threshold will be masked. For the sake of the demonstration I'll use a lower masking ratio than our model.

In [4]:
mask_ratio = 0.4 
rand = torch.rand(batch, num_genes)
rand

tensor([[0.0783, 0.4956, 0.6231, 0.4224, 0.2004, 0.0287, 0.5851, 0.6967],
        [0.1761, 0.2595, 0.7086, 0.5809, 0.0574, 0.7669, 0.8778, 0.2434]])

In [5]:
mask_idx = rand < mask_ratio #use probabilistic masking. it's not perfect but will work well over a large training run
mask_idx

tensor([[ True, False, False, False,  True,  True, False, False],
        [ True,  True, False, False,  True, False, False,  True]])

Now we'll apply the mask. I'll make a copy of our `x_values` so we can see how the masking changes. 

In [6]:
x_values = x_input.clone()
x_values[mask_idx] = 0.0
x_values.shape, x_values

(torch.Size([2, 8]),
 tensor([[0., 2., 2., 3., 0., 0., 2., 5.],
         [0., 0., 3., 4., 0., 5., 3., 0.]]))

## Forward Pass

We start by inserting in a channels dimension. Right now we just have 1 value per gene, but we'll represent each gene with many dimensions `embed_dim` to let the model learn different combinations of gene importance. We'll also use multiple `heads` which is basically a grouping of the embedding dimension channels so that combinations of them can learn different complex topics. 

In [7]:
embed_dim = 6
heads = 2

**Insert in the channels dimension**

In [8]:
x = x_values.unsqueeze(-1)
x.shape, x

(torch.Size([2, 8, 1]),
 tensor([[[0.],
          [2.],
          [2.],
          [3.],
          [0.],
          [0.],
          [2.],
          [5.]],
 
         [[0.],
          [0.],
          [3.],
          [4.],
          [0.],
          [5.],
          [3.],
          [0.]]]))

### Fourier FiLM gene encoding

Instead of just relying on gene expression counts, we want to do a Fourier FiLM based projection of the gene expression counts with learnable controls on the projection. 

![full_model_overview](../resources/v0_6/fourier_film_encoder.png)

We do this since we know that in biology, expression is typically non-linear where you have different expression plateaus including fully on or off. In this projection expression values are first scaled by learned gene embeddings and a scaler, then a random Fourier feature network generates expression-dependent FiLM parameters (gamma, beta) to further modulate the result based on expression level. The two paths give the model both a linear signal (expression * embedding) and a nonlinear one (Fourier features -> FiLM), combined into the final gene representation. The goal is to apply the following

$\mathbf{h}_g = \alpha \, x_g \, \mathbf{e}_g \odot (1 + \boldsymbol{\gamma}_g) + \boldsymbol{\beta}_g$

where

$[\boldsymbol{\gamma}_g, \boldsymbol{\beta}_g] = \mathrm{MLP}(\phi(\alpha' x_g))$

- $x_g$ is the expression value for gene $g$
- $\mathbf{e}_g$ is the learned gene embedding
- $\alpha, \alpha'$ are learned scalars
- $\phi$ is a random Fourier feature mapping
- $\odot$ is elementwise multiplication

While we use a multilayer perceptron (MLP), the FiLM portion is specifically the $\mathbf{h}_g = x * (1 + \gamma) + \beta$ pattern. This becomes an affine transformation where gamma and beta are conditioned on some input. 

#### $\alpha$ Gene Expression Count Scaling

We'll start with our gene expression count scaler. This will be a single value that we'll multiply against our scaled gene expression count embeddings. We use a linear layer here so that backprop can update this scaler. We'll first setup the learned scaler and multiply it by expression counts, after which we'll then use that to scale our gene embeddings. We'll initialize this scaler to `1.5` so you'll see that our initial expression values grow.

In [9]:
expr_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(expr_scaler.weight, 1.5)
expr_scaler.weight

Parameter containing:
tensor([[1.5000]], requires_grad=True)

In [10]:
scaled_x = expr_scaler(x)
scaled_x.shape, scaled_x

(torch.Size([2, 8, 1]),
 tensor([[[0.0000],
          [3.0000],
          [3.0000],
          [4.5000],
          [0.0000],
          [0.0000],
          [3.0000],
          [7.5000]],
 
         [[0.0000],
          [0.0000],
          [4.5000],
          [6.0000],
          [0.0000],
          [7.5000],
          [4.5000],
          [0.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\mathbf{e}_g$ Gene Embeddings

Now we'll initialize a representation of the gene embeddings and then scale them based on our scaled expression counts. You'll see that the masking extends across all embedding channels. Also, since we used an initialization where each channel has the same value, you'll see that the multiplication creates consistent channel entries per gene

In [11]:
# creates an incremental weight for easier following
vs, d = num_genes, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gene_embeddings = nn.Parameter(pattern)
gene_embeddings

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
        [0.7000, 0.7000, 0.7000, 0.7000, 0.7000, 0.7000],
        [0.8000, 0.8000, 0.8000, 0.8000, 0.8000, 0.8000]], requires_grad=True)

In [12]:
scaled_x = gene_embeddings.unsqueeze(0) * scaled_x
scaled_x

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
         [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
         [1.8000, 1.8000, 1.8000, 1.8000, 1.8000, 1.8000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [2.1000, 2.1000, 2.1000, 2.1000, 2.1000, 2.1000],
         [6.0000, 6.0000, 6.0000, 6.0000, 6.0000, 6.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [1.3500, 1.3500, 1.3500, 1.3500, 1.3500, 1.3500],
         [2.4000, 2.4000, 2.4000, 2.4000, 2.4000, 2.4000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [4.5000, 4.5000, 4.5000, 4.5000, 4.5000, 4.5000],
         [3.1500, 3.1500, 3.1500, 3.1500, 3.1500, 3.1500],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]]],
       grad_fn=<MulBackward0>)

#### $\alpha'$ Fourier Scaler 

Now we move on to the Fourier half of the calculation. We'll start by calculating the scaler. Again we use a linear here so that it represents a learnable parameter that will drift during backprop. To make sure the values diverge from the expression scaler, I'll use a different value `0.5`. When applied, you can see our initial expression values are cut in half. 

In [13]:
fourier_input_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(fourier_input_scaler.weight, 0.5)
fourier_input_scaler.weight

Parameter containing:
tensor([[0.5000]], requires_grad=True)

In [14]:
fourier_x = fourier_input_scaler(x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 1]),
 tensor([[[0.0000],
          [1.0000],
          [1.0000],
          [1.5000],
          [0.0000],
          [0.0000],
          [1.0000],
          [2.5000]],
 
         [[0.0000],
          [0.0000],
          [1.5000],
          [2.0000],
          [0.0000],
          [2.5000],
          [1.5000],
          [0.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\phi$ Fourier Projection  
Now we'll do the Fourier projection. This step projects the scaled expression counts through a fixed random matrix `fp_scaler`. This random matrix is half the size since we take the sin and cos of the result and then concatenate them together to produce a full-dimensional embedding. 

This step maps the continuous expression level into a rich high-dimensional representation where nearby values have similar features allowing the model to learn representations from the different plateaus of expression levels. The `gaussian_scale` scaler is a tunable hyperparameter that can quickly improve and degrade model performance.

Since we're working with sine/cosine we have to think of the periodicity. We first will take our expression and spread them across a full sin/cosine cycle, which, if you remember your trig, is $2\pi$. This normalizes the input so that a unit change in the input corresponds to one full cycle of sin/cos. If we did not do this, the random projection matrix alone controls the frequency, increasing the fragility of the projection.

**Numeric computing** one thing to remember is that we're working with fixed precision. Because of this, when we use irrational numbers like $\pi$, we must store a numeric representation meaning only a certain precision is stored. This means that while we should expect integer representations for multiples of $\pi$, we actually sometimes get infinitesimals instead. While it's not the most precise, this randomly introduced noise will just act as a mini-bias to our model. Take the below example, for `sin` we should see `[0,1,0,-1,0]` and for `cos` we should see `[1,0,-1,0,1]` but instead we see some values not quite there. Keep this in mind as we'll see this numeric computing error introduced in our notebook. 

In [15]:
examp_array = torch.tensor([0,0.25,0.5,0.75,1]).float()
torch.sin(examp_array*2*np.pi), torch.cos(examp_array*2*np.pi)

(tensor([ 0.0000e+00,  1.0000e+00, -8.7423e-08, -1.0000e+00,  1.7485e-07]),
 tensor([ 1.0000e+00, -4.3711e-08, -1.0000e+00,  1.1925e-08,  1.0000e+00]))

In [16]:
x_fp = (2 * np.pi * fourier_x)
x_fp

tensor([[[ 0.0000],
         [ 6.2832],
         [ 6.2832],
         [ 9.4248],
         [ 0.0000],
         [ 0.0000],
         [ 6.2832],
         [15.7080]],

        [[ 0.0000],
         [ 0.0000],
         [ 9.4248],
         [12.5664],
         [ 0.0000],
         [15.7080],
         [ 9.4248],
         [ 0.0000]]], grad_fn=<MulBackward0>)

Now that we've scaled our expression counts by $2\pi$, we'll inject half of the channels. As part of this injection, we use random initiated noise for each channel and a tunable hyperparameter scaler `gaussian_scale` to increment the noise. For learning, we'll keep our initialization preset.

That said, you'll see that we do not use gradients as this is not learnable. Our goal with the random Fourier features is that a fixed random projection is theoretically sufficient to approximate a shift-invariant kernel (we care about differences in expression values, not the actual numeric number). If we made it learnable, the network would likely collapse the frequencies to overfit or degenerate, defeating the purpose of providing a diverse multi-frequency basis. The nonlinear expressiveness of this layer comes downstream with the film generation MLP that processes the Fourier features. That layer is learnable.

In [17]:
gaussian_scale = 2.0
fp_scaler = nn.Parameter(torch.tensor([[0.5,1.0,1.5]])* gaussian_scale, requires_grad=False)
fp_scaler

Parameter containing:
tensor([[1., 2., 3.]])

In [18]:
x_fp = x_fp @ fp_scaler
x_fp.shape, x_fp

(torch.Size([2, 8, 3]),
 tensor([[[ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [ 6.2832, 12.5664, 18.8496],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [15.7080, 31.4159, 47.1239]],
 
         [[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 9.4248, 18.8496, 28.2743],
          [12.5664, 25.1327, 37.6991],
          [ 0.0000,  0.0000,  0.0000],
          [15.7080, 31.4159, 47.1239],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000]]], grad_fn=<UnsafeViewBackward0>))

**Sin/Cos**

Now we're ready for our radial extraction to take the sine and cosine. As a reminder, because we're dealing with numeric computing, instead of nice clean integers we'll see that we have infinitesimals introduced. Interestingly, this is more prevalent in sine vs cosine bringing us back up to our embedding dimension size. 

In [19]:
x_fp_sin = torch.sin(x_fp)
x_fp_sin

tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06]],

        [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 3.4969e-07,  6.9938e-07,  9.5399e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]]], grad_fn=<SinBackward0>)

In [20]:
x_fp_cos = torch.cos(x_fp)
x_fp_cos

tensor([[[ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.]],

        [[ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.]]], grad_fn=<CosBackward0>)

In [21]:
fourier_x = torch.cat([x_fp_sin, x_fp_cos], dim=-1)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-2.3850e-08,  4.7700e-08, -7.1549e-08, -1.0000e+00,  1.0000e+00,
           -1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-6.7553e-07,  1.3511e-06, -3.9339e-06, -1.0000e+00,  1.0000e+00,
           -1.0000e+00]],
 
         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  

#### $\mathrm{MLP}$ Multilayer perceptron  

Now we're ready to add the nonlinear expressiveness for the film generator by using a MLP. The MLP will provide a learnable linear layer, a nonlinearity, and a final upward projection into a doubling to create our $\gamma$ and $\beta$ for our FiLM calculation. These learnable layers are what allow the model to decide how much and which parts of the Fourier Projection we want to include with our initial scaled expression. 

**MLP - linear 1** We'll first start with a single linear layer that allows the model to decide how much each channel learned should interact with the other. Recall that half our channels are sine, and half are cosine, so this allows the model to mix the two radial projections. 

*You'll notice that near 0 values here are washed out*

In [22]:
mlp_l1 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(mlp_l1.weight, .25)
nn.init.constant_(mlp_l1.bias, 0.0)
mlp_l1.weight

Parameter containing:
tensor([[0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500]], requires_grad=True)

In [23]:
fourier_x = mlp_l1(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500]],
 
         [[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0

**MLP - GELU nonlinearity** Now we're ready for our non-linearity. The [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html) function is approximately linear above 1 and pulls most values below -2 to 0. Between -2 and 0, most values are pulled closer to 0 and there's a slight non-linearity between 0 and 1. Since we used sine/cosine values, most of our values at this point should be between 0 and 1 so we'll benefit from the non-linear portion with a cap on highly negative values. 

In [24]:
mlp_gelu = nn.GELU()

fourier_x = mlp_gelu(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003]],
 
         [[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0

**MLP - linear 2** Now we'll scale up to 2x our embedding size since we need to provide values for our two variables in our FiLM projection. We use a learnable linear scale up layer so that the model can learn how to split values across the two variables. For our initialization we'll increment the second half to be twice the first half to show the differences. 

In [25]:
mlp_l2 = nn.Linear(embed_dim, embed_dim*2)
nn.init.constant_(mlp_l2.weight[:embed_dim, :], 0.5)
nn.init.constant_(mlp_l2.weight[embed_dim:, :], 1.0)
nn.init.constant_(mlp_l2.bias, 0.0)
mlp_l2.weight.shape, mlp_l2.weight

(torch.Size([12, 6]),
 Parameter containing:
 tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]], requires_grad=True))

In [26]:
fourier_x = mlp_l2(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 12]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.6019,
           -0.6019, -0.6019, -0.6019, -0.6019, -0.6019],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\gamma_g, \beta_g$ FiLM components 
Now that we have the MLP output, we're ready to build our FiLM variables. Recall that for FiLM we calculate $\mathbf{h}_g = x * (1 + \gamma) + \beta$ where $x$ will be the weighted scaler of the expression counts. To get $\gamma$ and $\beta$ we simply split the output of the MLP. Recall that we built it so that half of the MLP output was doubled so when we split, we should see that $\beta$ is double $\gamma$.

In [27]:
gamma, beta = torch.chunk(fourier_x, 2, dim=-1)
gamma.shape, gamma, beta.shape, beta

(torch.Size([2, 8, 6]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010]],
 
         [[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\mathbf{h}_g$ FiLM based gene expression representation 

Now we're ready for the final FiLM calculation. FiLM can be thought of as summing two weighted parts, in our case a scaled version of the expression counts and a radial projection of the expression counts. What's interesting is that the FiLM formula pushes the radial projection both as a scaler to the initial counts and a bias similar to $x = mx + b$. Ultimately we've given the model the ability to learn how to upscale the original counts, shift the counts based on the radial projection, and add/subtract the counts based on the radial projection. All of this allows the model to learn a more complex landscape to scale gene embeddings by the expression beyond the typical linear scaling where 2 expression counts mean double 1 expression. 

In [28]:
x = scaled_x * (1.0 + gamma) + beta

x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 5.1242,  5.1242,  5.1242,  5.1242,  5.1242,  5.1242],
          [ 5.9463,  5.9463,  5.9463,  5.9463,  5.9463,  5.9463],
          [ 0.6563,  0.6563,  0.6563,  0.6563,  0.6563,  0.6563],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 9.2344,  9.2344,  9.2344,  9.2344,  9.2344,  9.2344],
          [ 3.5922,  3.5922,  3.5922,  3.5922,  3.5922,  3.5922]],
 
         [[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 0.3417,  0.3417,  0.3417,  0.3417,  0.3417,  0.3417],
          [10.0564, 10.0564, 10.0564, 10.0564, 10.0564, 10.0564],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 2.5437,  2.5437,  2.5437,  2.5437,  2.5437,  2.5437],
          [ 1.6000,  1.6000,  1.6000,  1.6000,  1

#### Remask with learned mask

Now we need to reintroduce the masking back but, instead of a 0 value, we want to actually let the model learn a mask token. We learn a mask token because the model needs to distinguish "this gene is masked and I need to predict it" from "this gene has zero expression." If the mask token were fixed (e.g. all zeros), it would be indistinguishable from a zero-expression gene's representation after the Fourier FiLM encoding. A learned token lets the model settle on a representation that optimally signals "predict me" to the downstream transformer blocks. Ultimately we're reapplying the masking at this point mainly so that we do our expression encoding cleanly first, and then mask after. Since our mask token is learnable, instead of a common `-1` hardcode, the initialization in the model will use random learnable values. What we'll do in our example is use `-11` so it really sticks out. These values will change during backprop as the model learns what a good token is to prompt it that it needs replacement. 

*As a reminder, masking only happens on the context encoder, and not the target encoder during our models forward pass*

In [29]:
mask_token = nn.Parameter(torch.randn(embed_dim) * 0.02)
nn.init.constant_(mask_token, -11)
mask_token

Parameter containing:
tensor([-11., -11., -11., -11., -11., -11.], requires_grad=True)

In [30]:
x = torch.where(mask_idx.unsqueeze(-1), mask_token, x)
x

tensor([[[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  5.1242,   5.1242,   5.1242,   5.1242,   5.1242,   5.1242],
         [  5.9463,   5.9463,   5.9463,   5.9463,   5.9463,   5.9463],
         [  0.6563,   0.6563,   0.6563,   0.6563,   0.6563,   0.6563],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  9.2344,   9.2344,   9.2344,   9.2344,   9.2344,   9.2344],
         [  3.5922,   3.5922,   3.5922,   3.5922,   3.5922,   3.5922]],

        [[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  0.3417,   0.3417,   0.3417,   0.3417,   0.3417,   0.3417],
         [ 10.0564,  10.0564,  10.0564,  10.0564,  10.0564,  10.0564],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  2.5437,   2.5437,   2.5437,   2.5437,   2.5437,   2.5437],
    

### Total count injection

Now we'll want to add the total count in. In our data prep, we normalize all cell expression counts to the same total count. This is great since Perturb-seq is relativistic in its data, but this normalization has one drawback: cells with abnormally high or low expression totals compared to their peers lose the signal. In particular the abnormally low is our biggest concern as a perturbation that makes a cell barely viable may have very low across the board expression that gets amplified when you bring it up to our normalized count. To avoid this we add in a learnable weight to the total expression count so that the model can learn expression levels.

The total count is multiplied across the embedding dimensions with a learnable weight and then summed to our gene expression projections. This allows the total count to act almost like a bias term. 

We start by taking our total count, injecting the embedding dimensions, and then shaping it to match our embedding counts. 

In [31]:
x_total_ct = total_counts.unsqueeze(-1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1]),
 tensor([[2.],
         [4.]]))

In [32]:
total_count_proj = nn.Linear(1, embed_dim)
nn.init.constant_(total_count_proj.weight, 0.1)
nn.init.zeros_(total_count_proj.bias)
total_count_proj.weight

Parameter containing:
tensor([[0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000]], requires_grad=True)

In [33]:
x_total_ct = total_count_proj(x_total_ct)
x_total_ct = x_total_ct.unsqueeze(1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1, 6]),
 tensor([[[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]],
 
         [[0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000]]],
        grad_fn=<UnsqueezeBackward0>))

### Unified representation of the cell state

Now that we have an embedding representation of both the gene expression and total count, we're ready to sum them for a single representation of the cell state. Now it will be ready for a cell state block

In [34]:
x = x + x_total_ct
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[-10.8000, -10.8000, -10.8000, -10.8000, -10.8000, -10.8000],
          [  5.3242,   5.3242,   5.3242,   5.3242,   5.3242,   5.3242],
          [  6.1463,   6.1463,   6.1463,   6.1463,   6.1463,   6.1463],
          [  0.8563,   0.8563,   0.8563,   0.8563,   0.8563,   0.8563],
          [-10.8000, -10.8000, -10.8000, -10.8000, -10.8000, -10.8000],
          [-10.8000, -10.8000, -10.8000, -10.8000, -10.8000, -10.8000],
          [  9.4344,   9.4344,   9.4344,   9.4344,   9.4344,   9.4344],
          [  3.7922,   3.7922,   3.7922,   3.7922,   3.7922,   3.7922]],
 
         [[-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [  0.7417,   0.7417,   0.7417,   0.7417,   0.7417,   0.7417],
          [ 10.4564,  10.4564,  10.4564,  10.4564,  10.4564,  10.4564],
          [-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [  2.9437,   2.9437,   2.94

### Transformer Block 
The transformer layer in our model consists of 4 layers:
1. RMS normalization
2. Gated linear self-attention
3. RMS normalization
4. SwiGLU

Residual connections provide gradient bypassing around the attention and SwiGLU layers. This set of 4 units is repeated based on how many layers are configured. 

#### RMSNorm 1

For our modern transformer, we use root mean square normalization, or RMSNorm. RMSNorm calculates the following: 
$$
y = \frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot \gamma
$$

The main reason we use RMSNorm is that it executes faster and uses less memory than standard layer normalization. This efficiency is achieved by entirely removing the mean-centering calculation, which reduces the total number of arithmetic operations and hardware synchronization steps. Recall that normalization is primarily used for large scale training stability (preventing gradient explosion/vanishing). For deep architectures like ours, the mean of the pre-activation inputs naturally stays close to zero during training so removing the mean-centering operation preserves the critical variance-bounding effect. Also the model learns to absorb any minor activation shifts into the subsequent linear weights or the learned affine parameters. Because of this, we're able to use a more efficient normalization.

Since we reuse RMSNorm 3 times, we'll create a class for it. In the class you can see that we split out the calculation, first limiting the precision of X, then computing $\frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}}$, and finally adding the per-channel weights $\cdot \gamma$.

Since our entries have identical values across all channels for a gene, the gene becomes uniform +/-1 depending on the sign (the value is the average so it becomes 1)

In [35]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight).type_as(x)

In [36]:
rms1 = RMSNorm(embed_dim)

rms1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [37]:
x_norm = rms1(x)
x_norm.shape, x_norm

(torch.Size([2, 8, 6]),
 tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000]],
 
         [[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1

#### Multi-Headed Gated Linear Attention
Since our current token size is 10000 and we know it's going to grow to 20,000+ when we add all genes and other cell state representations, we want to avoid building the attention matrix in memory. To avoid this we chose linear attention. Linear attention foregoes materializing the attention matrix. In the encoder, we do self attention allowing each gene to build a relationship with any other gene. Our attention will be multi-headed, meaning we'll split up our embedding space across the heads allowing them each to learn different complex representations. Finally we'll also add gating at the end to allow the model to determine how much the attention should impact each specific gene. Linear attention still incorporates the query, key, and value, but removes the need for softmax. We calculate

$$\begin{aligned}
\text{elu}(x) &= \begin{cases} x & \text{if } x > 0 \\ \alpha (e^x - 1) & \text{if } x \le 0 \end{cases} \\
\\
Q &= \text{elu}(x W_q^\top + b_q) + 1.0 \\
K &= \text{elu}(x W_k^\top + b_k) + 1.0 \\
V &= x W_v^\top + b_v \\
\\
\text{Attn} &= \frac{Q K^\top V}{Q (\sum_{j=1}^T K_j)^\top + \epsilon} \\
\text{Gate} &= \sigma(x W_{gate}^\top + b_{gate}) \\
\\
y &= \left( \text{Gate} \odot \text{Attn} \right) W_c^\top + b_c
\end{aligned}$$

In [38]:
B, T_q, C = x_norm.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 8, 6, 3)

**Self attention** 

For our encoder, the linear attention is self attention, meaning it allows for each token (gene) to build a relationship with the other tokens. Typically this can be extremely memory expensive as we'd make a TxT matrix in memory. As you'll see, with linear attention we do not need to do that. You might now ask: why even include this variable `kv_input`. This is because we built our linear attention to be both self attention and cross attention. Cross attention is when you're building a relationship between two different inputs. 

In [39]:
kv_input = x_norm
kv_input

tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000]],

        [[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0

In [40]:
T_kv = kv_input.size(1)
T_kv

8

**Query** 

Let's start with our query. The query is the current position's "search request." Every layer and head issues queries that can look for relationships. Our query first calculates a linear weight $Q=x_{norm}W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [41]:
q_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj.weight, -0.1)
nn.init.constant_(q_proj.bias, 0)
q_proj.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [42]:
q = q_proj(x_norm).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 8, 3]),
 tensor([[[[ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000]],
 
          [[ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000]]],
 
 
         [[[ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000]],
 
 

In [43]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 8, 3]),
 tensor([[[[1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488]],
 
          [[1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488]]],
 
 
         [[[1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000]],
 
          [[1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],

**Key** 

Next, we calculate the key. The key acts as a matching tag/address for each allowed token. It is compared with the query to produce relevance scores. If this was cross-attention and kv was provided, K would be based on it. Since it's self attention, we again project the normalized X allowing the model to build the second half of the cross. 

We again calculate a linear weight $K=x_{kv\_input}\cdot W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [44]:
k_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj.weight, 0.2)
nn.init.constant_(k_proj.bias, 0)
k_proj.weight

Parameter containing:
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]], requires_grad=True)

In [45]:
k = k_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 8, 3]),
 tensor([[[[-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000]],
 
          [[-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000]]],
 
 
         [[[-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000]],
 
 

In [46]:
k = F.elu(k) + 1.0
k

tensor([[[[0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000]],

         [[0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000]]],


        [[[0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012]],

         [[0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2

**Value** 

Next, we calculate the value. The value is the payload you actually mix in once something matches. It's a learned projection of the token's representation so the model can copy the right kind of information. Similarly, since this is self attention, V will be based on X. 

We again calculate a linear weight $V=x_{kv\_input}\cdot W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

For V we do not use ELU+1 since the QK will act on V. 

In [47]:
v_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj.weight, 1.0)
nn.init.constant_(v_proj.bias, 0)
v_proj.weight

Parameter containing:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]], requires_grad=True)

In [48]:
v = v_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 8, 3]),
 tensor([[[[-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000]],
 
          [[-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000]]],
 
 
         [[[-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000]],
 
 

**Normalization denominator** 

In attention, when using softmax, attention values become probabilities and sum to 1. With linear attention, we need to manually do this normalization.

In softmax attention, the softmax inherently normalizes so weights sum to 1. Linear attention doesn't have that, so we need to create the denominator $z$. We'll do this by summing the tokens per embedding dimension resulting in a $[B,Heads,C_{Heads},1]$ dimension that we can then use in the denominator of our attention calculation

In [49]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 3, 1]),
 tensor([[[[11.9036],
           [11.9036],
           [11.9036]],
 
          [[11.9036],
           [11.9036],
           [11.9036]]],
 
 
         [[[10.0048],
           [10.0048],
           [10.0048]],
 
          [[10.0048],
           [10.0048],
           [10.0048]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator** 

We're now ready to calculate the rest of the denominator. The denominator is the sum of attention weights for query across all keys. Each query gets its own normalizing constant, so queries attending to high-magnitude keys don't get inflated outputs. We add a final epsilon to ensure the denominator is not zero. The result from this is that we sum across the head dimensions creating a $[B,Heads,T,1]$ result

In [50]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 8, 1]),
 tensor([[[[0.0175],
           [0.0510],
           [0.0510],
           [0.0510],
           [0.0175],
           [0.0175],
           [0.0510],
           [0.0510]],
 
          [[0.0175],
           [0.0510],
           [0.0510],
           [0.0510],
           [0.0175],
           [0.0175],
           [0.0510],
           [0.0510]]],
 
 
         [[[0.0208],
           [0.0208],
           [0.0607],
           [0.0607],
           [0.0208],
           [0.0607],
           [0.0607],
           [0.0208]],
 
          [[0.0208],
           [0.0208],
           [0.0607],
           [0.0607],
           [0.0208],
           [0.0607],
           [0.0607],
           [0.0208]]]], grad_fn=<MulBackward0>))

**Numerator**

Now we're ready to complete our numerator. This is just a matter of multiplying Q, K, and V. We'll need to transpose K to get the interaction between the query and key to then multiply against the value. 

Since we're looking to save memory, we actually first multiply the key and value to create head dimension matrixes, and then multiply by the query. This order of operations is part of what saves memory. 

In [51]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 3, 3]),
 tensor([[[[60.5785, 60.5785, 60.5785],
           [60.5785, 60.5785, 60.5785],
           [60.5785, 60.5785, 60.5785]],
 
          [[60.5785, 60.5785, 60.5785],
           [60.5785, 60.5785, 60.5785],
           [60.5785, 60.5785, 60.5785]]],
 
 
         [[[45.5713, 45.5713, 45.5713],
           [45.5713, 45.5713, 45.5713],
           [45.5713, 45.5713, 45.5713]],
 
          [[45.5713, 45.5713, 45.5713],
           [45.5713, 45.5713, 45.5713],
           [45.5713, 45.5713, 45.5713]]]], grad_fn=<UnsafeViewBackward0>))

In [52]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 8, 3]),
 tensor([[[[290.7767, 290.7767, 290.7767],
           [ 99.7385,  99.7385,  99.7385],
           [ 99.7385,  99.7385,  99.7385],
           [ 99.7386,  99.7386,  99.7386],
           [290.7767, 290.7767, 290.7767],
           [290.7767, 290.7767, 290.7767],
           [ 99.7385,  99.7385,  99.7385],
           [ 99.7385,  99.7385,  99.7385]],
 
          [[290.7767, 290.7767, 290.7767],
           [ 99.7385,  99.7385,  99.7385],
           [ 99.7385,  99.7385,  99.7385],
           [ 99.7386,  99.7386,  99.7386],
           [290.7767, 290.7767, 290.7767],
           [290.7767, 290.7767, 290.7767],
           [ 99.7385,  99.7385,  99.7385],
           [ 99.7385,  99.7385,  99.7385]]],
 
 
         [[[218.7423, 218.7423, 218.7423],
           [218.7423, 218.7423, 218.7423],
           [ 75.0303,  75.0303,  75.0303],
           [ 75.0302,  75.0302,  75.0302],
           [218.7423, 218.7423, 218.7423],
           [ 75.0302,  75.0302,  75.0302],
           [ 75.03

**Linear Attention** 

Now we're ready to normalize our numerator by the denominator. You'll see that because of our extremely consistent values and initialization we're ending up with very consistent values by example even across heads. 

In [53]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 8, 3]),
 tensor([[[[5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891]],
 
          [[5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891],
           [5.0891, 5.0891, 5.0891]]],
 
 
         [[[4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550]],
 
          [[4.5550, 4.5550, 4.5550],
           [4.5550, 4.5550, 4.5550],

**Collapse heads** 

We now can bring our heads back together. We have to undo our head splitting. First we flip our heads and tokens (genes) so that we have a $[B,T,Heads,C_{Heads}]$ and then we collapse the head and head embeddings to end up with $[B,T,C]$

In [54]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 8, 6]),
 tensor([[[5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891],
          [5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891],
          [5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891],
          [5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891],
          [5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891],
          [5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891],
          [5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891],
          [5.0891, 5.0891, 5.0891, 5.0891, 5.0891, 5.0891]],
 
         [[4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550],
          [4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550],
          [4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550],
          [4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550],
          [4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550],
          [4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550],
          [4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550],
          [4.5550, 4.5550, 4.5550, 4.5550, 4.5550, 4.5550]]],
        gra

**Gating** 

Now we're ready to add sigmoid based gating on top of our attention. This gating uses the Hadamard product of the gate and the attention, which lets the model learn to suppress or pass through attention output per dimension. This allows the model to decide how much of the attention result to actually use for each dimension. We use a learned linear weight and sigmoid to pull gating values between 0 and 1, allowing the model to turn down specific values. 

In [55]:
gate = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gate.weight = nn.Parameter(pattern)
gate.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000]], requires_grad=True)

In [56]:
y = torch.sigmoid(gate(x_norm)) * y
y.shape, y

(torch.Size([2, 8, 6]),
 tensor([[[2.2188, 1.5103, 0.7194, 0.4398, 0.3162, 0.1549],
          [3.6622, 4.1886, 4.3647, 4.6818, 4.9056, 4.9708],
          [3.6622, 4.1886, 4.3647, 4.6818, 4.9056, 4.9708],
          [3.6622, 4.1886, 4.3647, 4.6818, 4.9056, 4.9708],
          [2.2188, 1.5103, 0.7194, 0.4398, 0.3162, 0.1549],
          [2.2188, 1.5103, 0.7194, 0.4398, 0.3162, 0.1549],
          [3.6622, 4.1886, 4.3647, 4.6818, 4.9056, 4.9708],
          [3.6622, 4.1886, 4.3647, 4.6818, 4.9056, 4.9708]],
 
         [[1.9859, 1.3517, 0.6439, 0.3937, 0.2830, 0.1386],
          [1.9859, 1.3517, 0.6439, 0.3937, 0.2830, 0.1386],
          [3.2778, 3.7490, 3.9066, 4.1904, 4.3907, 4.4491],
          [3.2778, 3.7490, 3.9066, 4.1904, 4.3907, 4.4491],
          [1.9859, 1.3517, 0.6439, 0.3937, 0.2830, 0.1386],
          [3.2778, 3.7490, 3.9066, 4.1904, 4.3907, 4.4491],
          [3.2778, 3.7490, 3.9066, 4.1904, 4.3907, 4.4491],
          [1.9859, 1.3517, 0.6439, 0.3937, 0.2830, 0.1386]]],
        gra

**Cross-head final projection** 

Finally, we will now project the gated attention matrix on another final linear layer. This allows the model to learn how to combine information across the different heads. 

In [57]:
c_proj = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.01*cols)  

c_proj.weight = nn.Parameter(pattern)
c_proj.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300, 0.1300, 0.1300],
        [0.1400, 0.1400, 0.1400, 0.1400, 0.1400, 0.1400],
        [0.1500, 0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], requires_grad=True)

In [58]:
x_attn = c_proj(y)
x_attn.shape, x_attn

(torch.Size([2, 8, 6]),
 tensor([[[0.7425, 0.3425, 0.5842, 0.5182, 0.7168, 0.7645],
          [2.8839, 2.6981, 3.1539, 3.3020, 3.7148, 3.9766],
          [2.8839, 2.6981, 3.1539, 3.3020, 3.7148, 3.9766],
          [2.8839, 2.6981, 3.1539, 3.3020, 3.7148, 3.9766],
          [0.7425, 0.3425, 0.5842, 0.5182, 0.7168, 0.7645],
          [0.7425, 0.3425, 0.5842, 0.5182, 0.7168, 0.7645],
          [2.8839, 2.6981, 3.1539, 3.3020, 3.7148, 3.9766],
          [2.8839, 2.6981, 3.1539, 3.3020, 3.7148, 3.9766]],
 
         [[0.6862, 0.2806, 0.5167, 0.4450, 0.6380, 0.6801],
          [0.6862, 0.2806, 0.5167, 0.4450, 0.6380, 0.6801],
          [2.6029, 2.3890, 2.8167, 2.9367, 3.3214, 3.5551],
          [2.6029, 2.3890, 2.8167, 2.9367, 3.3214, 3.5551],
          [0.6862, 0.2806, 0.5167, 0.4450, 0.6380, 0.6801],
          [2.6029, 2.3890, 2.8167, 2.9367, 3.3214, 3.5551],
          [2.6029, 2.3890, 2.8167, 2.9367, 3.3214, 3.5551],
          [0.6862, 0.2806, 0.5167, 0.4450, 0.6380, 0.6801]]],
        gra

#### Residual Connection

We now have our gated linear attention calculated and will use a residual connection to allow gradients to bypass the attention matrix. With this you'll see how much larger the residual connection impact is on our output compared to our attention. 

In [59]:
x = x + x_attn
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[-10.0575, -10.4575, -10.2158, -10.2818, -10.0832, -10.0355],
          [  8.2081,   8.0223,   8.4782,   8.6263,   9.0390,   9.3009],
          [  9.0302,   8.8443,   9.3002,   9.4483,   9.8611,  10.1229],
          [  3.7402,   3.5544,   4.0103,   4.1583,   4.5711,   4.8330],
          [-10.0575, -10.4575, -10.2158, -10.2818, -10.0832, -10.0355],
          [-10.0575, -10.4575, -10.2158, -10.2818, -10.0832, -10.0355],
          [ 12.3183,  12.1325,  12.5883,  12.7364,  13.1492,  13.4110],
          [  6.6761,   6.4903,   6.9462,   7.0943,   7.5070,   7.7689]],
 
         [[ -9.9138, -10.3194, -10.0833, -10.1550,  -9.9620,  -9.9199],
          [ -9.9138, -10.3194, -10.0833, -10.1550,  -9.9620,  -9.9199],
          [  3.3446,   3.1307,   3.5585,   3.6785,   4.0631,   4.2969],
          [ 13.0593,  12.8454,  13.2731,  13.3931,  13.7778,  14.0115],
          [ -9.9138, -10.3194, -10.0833, -10.1550,  -9.9620,  -9.9199],
          [  5.5466,   5.3327,   5.76

#### RMSNorm 2
Now that we've calculated the attention, we'll do another round of normalization. We'll use RMSNorm again. 

In [60]:
rms2= RMSNorm(embed_dim)
rms2.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [61]:
x_norm2 = rms2(x)
x_norm2.shape, x_norm2

(torch.Size([2, 8, 6]),
 tensor([[[-0.9870, -1.0263, -1.0026, -1.0090, -0.9896, -0.9849],
          [ 0.9518,  0.9302,  0.9831,  1.0003,  1.0481,  1.0785],
          [ 0.9561,  0.9364,  0.9847,  1.0004,  1.0441,  1.0718],
          [ 0.8973,  0.8527,  0.9621,  0.9976,  1.0966,  1.1594],
          [-0.9870, -1.0263, -1.0026, -1.0090, -0.9896, -0.9849],
          [-0.9870, -1.0263, -1.0026, -1.0090, -0.9896, -0.9849],
          [ 0.9676,  0.9530,  0.9888,  1.0005,  1.0329,  1.0535],
          [ 0.9410,  0.9148,  0.9791,  1.0000,  1.0582,  1.0951]],
 
         [[-0.9855, -1.0258, -1.0023, -1.0094, -0.9903, -0.9861],
          [-0.9855, -1.0258, -1.0023, -1.0094, -0.9903, -0.9861],
          [ 0.9039,  0.8461,  0.9617,  0.9941,  1.0981,  1.1612],
          [ 0.9746,  0.9587,  0.9906,  0.9995,  1.0282,  1.0457],
          [-0.9855, -1.0258, -1.0023, -1.0094, -0.9903, -0.9861],
          [ 0.9410,  0.9047,  0.9773,  0.9977,  1.0629,  1.1026],
          [ 0.9293,  0.8861,  0.9725,  0.9967,  1

#### SwiGLU 

Next, we'll introduce our scaling nonlinearity layers. Traditionally this was a MLP, but we replaced it with a swish-gated linear unit, or SwiGLU. SwiGLU replaces the single linear transform in a standard MLP with a gated pathway with a result as follows:

$$\begin{aligned}
\text{SiLU}(Z) &= Z \odot \sigma(Z) \\
\text{gate} &= \text{SiLU}(x W_g^\top) \\
H &= x W_u^\top \\
y &= (\text{gate} \odot H) W_d^\top
\end{aligned}$$

The Hadamard product based gating lets the network learn to selectively amplify or suppress features before the final projection, giving it more expressive power per parameter than a standard two-layer MLP with ReLU/GELU at the cost of some extra compute.

*You'll notice that SwiGLU has 3 weight matrices $W_g, W_u, W_d$, instead of the typical 2 we use in MLP. In production code, you might see helper functions that convert MLP ratios to SwiGLU ratios using 2/3 multiples to maintain the number of parameters*

In [62]:
hidden_dim = 8

**Gate** 

We'll start by initializing our gating weight $W_g$ and calculating our gate. The gate will need to scale up to our hidden dimension as it will multiply directly against our weighted input. 

In [63]:
wg = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wg.weight, 0.5)
wg.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [64]:
xwg = wg(x_norm2)
xwg.shape, xwg

(torch.Size([2, 8, 8]),
 tensor([[[-2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997,
           -2.9997],
          [ 2.9960,  2.9960,  2.9960,  2.9960,  2.9960,  2.9960,  2.9960,
            2.9960],
          [ 2.9967,  2.9967,  2.9967,  2.9967,  2.9967,  2.9967,  2.9967,
            2.9967],
          [ 2.9829,  2.9829,  2.9829,  2.9829,  2.9829,  2.9829,  2.9829,
            2.9829],
          [-2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997,
           -2.9997],
          [-2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997,
           -2.9997],
          [ 2.9982,  2.9982,  2.9982,  2.9982,  2.9982,  2.9982,  2.9982,
            2.9982],
          [ 2.9941,  2.9941,  2.9941,  2.9941,  2.9941,  2.9941,  2.9941,
            2.9941]],
 
         [[-2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997,
           -2.9997],
          [-2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997, -2.9997,
           -2.9997],
          [ 2.9825,  2.

**SiLU Non-Linearity**

SiLU will pull our negative values closer to zero. Values above 1 will remain almost linear. When combined with the learned weights, you can quickly see how this becomes a gate. 

In [65]:
xwg = F.silu(xwg)
xwg.shape, xwg

(torch.Size([2, 8, 8]),
 tensor([[[-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [ 2.8534,  2.8534,  2.8534,  2.8534,  2.8534,  2.8534,  2.8534,
            2.8534],
          [ 2.8541,  2.8541,  2.8541,  2.8541,  2.8541,  2.8541,  2.8541,
            2.8541],
          [ 2.8391,  2.8391,  2.8391,  2.8391,  2.8391,  2.8391,  2.8391,
            2.8391],
          [-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [ 2.8557,  2.8557,  2.8557,  2.8557,  2.8557,  2.8557,  2.8557,
            2.8557],
          [ 2.8513,  2.8513,  2.8513,  2.8513,  2.8513,  2.8513,  2.8513,
            2.8513]],
 
         [[-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [ 2.8387,  2.

**Weighted Input** 

Now we'll need to scale up our input to the hidden dimension. We'll use a weighted layer $W_u$ allowing the model to determine how to use the different channels to create the new dimensions

In [66]:
wu = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wu.weight, -0.1)
wu.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [67]:
xwu = wu(x_norm2)
xwu.shape, xwu

(torch.Size([2, 8, 8]),
 tensor([[[ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [-0.5992, -0.5992, -0.5992, -0.5992, -0.5992, -0.5992, -0.5992,
           -0.5992],
          [-0.5993, -0.5993, -0.5993, -0.5993, -0.5993, -0.5993, -0.5993,
           -0.5993],
          [-0.5966, -0.5966, -0.5966, -0.5966, -0.5966, -0.5966, -0.5966,
           -0.5966],
          [ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [-0.5996, -0.5996, -0.5996, -0.5996, -0.5996, -0.5996, -0.5996,
           -0.5996],
          [-0.5988, -0.5988, -0.5988, -0.5988, -0.5988, -0.5988, -0.5988,
           -0.5988]],
 
         [[ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [-0.5965, -0.

**Apply Gate** Now we'll go ahead and apply the gate. We take the Hadamard product which allows the model to gate each value of the scaled up projection. 

In [68]:
xw = xwg * xwu
xw.shape, xw

(torch.Size([2, 8, 8]),
 tensor([[[-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-1.7098, -1.7098, -1.7098, -1.7098, -1.7098, -1.7098, -1.7098,
           -1.7098],
          [-1.7106, -1.7106, -1.7106, -1.7106, -1.7106, -1.7106, -1.7106,
           -1.7106],
          [-1.6937, -1.6937, -1.6937, -1.6937, -1.6937, -1.6937, -1.6937,
           -1.6937],
          [-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-1.7124, -1.7124, -1.7124, -1.7124, -1.7124, -1.7124, -1.7124,
           -1.7124],
          [-1.7074, -1.7074, -1.7074, -1.7074, -1.7074, -1.7074, -1.7074,
           -1.7074]],
 
         [[-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-1.6933, -1.

**Project Down** 

Now we need to project back down to our embedding dimension. We'll use a final weighted $W_d$ layer to determine how to project back down. This is similar to the final layer of an MLP. 

In [69]:
wd = nn.Linear(hidden_dim, embed_dim, bias=False)
nn.init.constant_(wd.weight, 0.33)
wd.weight

Parameter containing:
tensor([[0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300]],
       requires_grad=True)

In [70]:
xswig = wd(xw)
xswig.shape, xswig

(torch.Size([2, 8, 6]),
 tensor([[[-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.5137, -4.5137, -4.5137, -4.5137, -4.5137, -4.5137],
          [-4.5159, -4.5159, -4.5159, -4.5159, -4.5159, -4.5159],
          [-4.4715, -4.4715, -4.4715, -4.4715, -4.4715, -4.4715],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.5207, -4.5207, -4.5207, -4.5207, -4.5207, -4.5207],
          [-4.5076, -4.5076, -4.5076, -4.5076, -4.5076, -4.5076]],
 
         [[-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.4702, -4.4702, -4.4702, -4.4702, -4.4702, -4.4702],
          [-4.5223, -4.5223, -4.5223, -4.5223, -4.5223, -4.5223],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.5044, -4.5044, -4.5044, -4.5044, -4.5044, -4.5044],
          [-4.4951, -4.4951, -4.4951, -4.4951, -4

#### Residual Connection 2

We now have our SwiGLU based projection calculated and will use a residual connection to allow gradients to bypass the SwiGLU calculations. With this you'll see how much larger the residual connection impact is on our output compared to our SwiGLU output. 

In [71]:
x = x + xswig
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[-10.2829, -10.6829, -10.4412, -10.5072, -10.3086, -10.2609],
          [  3.6944,   3.5086,   3.9644,   4.1125,   4.5253,   4.7871],
          [  4.5143,   4.3285,   4.7843,   4.9324,   5.3452,   5.6070],
          [ -0.7313,  -0.9171,  -0.4612,  -0.3131,   0.0996,   0.3615],
          [-10.2829, -10.6829, -10.4412, -10.5072, -10.3086, -10.2609],
          [-10.2829, -10.6829, -10.4412, -10.5072, -10.3086, -10.2609],
          [  7.7975,   7.6117,   8.0676,   8.2157,   8.6285,   8.8903],
          [  2.1685,   1.9827,   2.4386,   2.5867,   2.9994,   3.2613]],
 
         [[-10.1392, -10.5448, -10.3087, -10.3804, -10.1873, -10.1453],
          [-10.1392, -10.5448, -10.3087, -10.3804, -10.1873, -10.1453],
          [ -1.1256,  -1.3395,  -0.9117,  -0.7917,  -0.4071,  -0.1733],
          [  8.5370,   8.3230,   8.7508,   8.8708,   9.2555,   9.4892],
          [-10.1392, -10.5448, -10.3087, -10.3804, -10.1873, -10.1453],
          [  1.0422,   0.8283,   1.25

### Final Layer Normalization
The previous layers can run sequentially for as many layers as is configured. The more layers, the "deeper" the network becomes. Once all the layers have completed, we're ready for a final normalization. Like previous normalizations this will pull our values together to focus on the variance. This output then becomes the latent representation of the context or target. 

In [72]:
rmsf = RMSNorm(embed_dim)
rmsf.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [73]:
x = rmsf(x)
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[-0.9873, -1.0257, -1.0025, -1.0089, -0.9898, -0.9852],
          [ 0.8961,  0.8510,  0.9616,  0.9975,  1.0976,  1.1611],
          [ 0.9141,  0.8764,  0.9687,  0.9987,  1.0823,  1.1353],
          [-1.3251, -1.6619, -0.8358, -0.5674,  0.1806,  0.6550],
          [-0.9873, -1.0257, -1.0025, -1.0089, -0.9898, -0.9852],
          [-0.9873, -1.0257, -1.0025, -1.0089, -0.9898, -0.9852],
          [ 0.9493,  0.9267,  0.9822,  1.0002,  1.0505,  1.0823],
          [ 0.8305,  0.7594,  0.9340,  0.9907,  1.1488,  1.2491]],
 
         [[-0.9858, -1.0252, -1.0023, -1.0092, -0.9905, -0.9864],
          [-0.9858, -1.0252, -1.0023, -1.0092, -0.9905, -0.9864],
          [-1.2697, -1.5110, -1.0285, -0.8931, -0.4592, -0.1955],
          [ 0.9614,  0.9373,  0.9854,  0.9990,  1.0423,  1.0686],
          [-0.9858, -1.0252, -1.0023, -1.0092, -0.9905, -0.9864],
          [ 0.7273,  0.5780,  0.8765,  0.9602,  1.2287,  1.3918],
          [ 0.1810, -0.1783,  0.5401,  0.7416,  1

# Latent Representation

We now have a latent representation of our input cell states. These representations can then be used in a number of ways. You'll notice that the shape still contains the batch, our context length which is the number of genes, and the embedding dimensions. Joint-embedding predictive architecture (JEPA) models work within this latent space. During our training loops we build energy based loss functions to help steer and shape this space through backprop updates.